In [1]:
# This script extracts raw features from waveform audio files
# All lines where "TRF for Alice EEG Dataset" pipeline (cited in the thesis) is followed are commented with: "TRF for Alice EEG Dataset"

from pathlib import Path
import numpy as np
import librosa
import eelbrain
import pickle

# Define paths that will be used throughout
DATA_ROOT = Path('/Users/zorkabozilovic/Desktop/PART1')
STIMULUS_DIR = DATA_ROOT / 'diliBach_wav_4dryad'

# Audio feature extraction parameters
AUDIO_SR = 44100
TARGET_SR = 100
N_MEL_BANDS = 8

In [2]:
# Extract acoustic features from WAV files

hop_length = AUDIO_SR // TARGET_SR  # 441 (exactly 100 frames/sec)
song_features = {}

for song_id in range(1, 11):
    
    wav_path = STIMULUS_DIR / f'{song_id}.wav'
    print(f'Song {song_id}/10')
   
    # Envelope (following "TRF for Alice EEG Dataset")
    wav_ndvar = eelbrain.load.wav(wav_path)
    envelope = wav_ndvar.envelope()
    envelope = eelbrain.resample(envelope, TARGET_SR)
    
    # Onsets (following "TRF for Alice EEG Dataset")
    onset = envelope.diff('time').clip(0)
    
    # Load with librosa
    y, sr = librosa.load(str(wav_path), sr=AUDIO_SR, mono=True)
   
    # Pitch (F0) with pyin
    f0, voiced_flag, voiced_prob = librosa.pyin (  # only f0 is used here
        y, sr=sr,
        fmin=librosa.note_to_hz('E3'), # set to 3 semitones below lowest note (G3) as safety margin
        fmax=librosa.note_to_hz('C7'), # set to 3 semitones above highest note (A6) as safety margin
        hop_length=hop_length,
    )
    # Interpolate through unvoiced NaN frames
    nan_mask = np.isnan(f0)
    if nan_mask.all():
        f0 = np.zeros_like(f0)
    elif nan_mask.any():
        voiced_indices = np.where(~nan_mask)[0]
        f0 = np.interp(np.arange(len(f0)), voiced_indices, f0[voiced_indices])
   
    # Spectral centroid
    cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)[0]
    
    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=hop_length, n_mels=N_MEL_BANDS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    freq_axis = eelbrain.Scalar('frequency', np.arange(N_MEL_BANDS))

    # Trim librosa features to match envelope length
    n = len(envelope)
    f0 = f0[:n]
    cent = cent[:n]
    mel_db = mel_db[:, :n]

    # Wrap librosa features in NDVars using envelope's time axis
    pitch_ndvar = eelbrain.NDVar(f0, (envelope.time,))
    cent_ndvar = eelbrain.NDVar(cent, (envelope.time,))
    mel_ndvar = eelbrain.NDVar(mel_db.T, (envelope.time, freq_axis))
  
    song_features[song_id] = {
        'envelope': envelope,
        'onsets': onset,
        'pitch': pitch_ndvar,
        'centroid': cent_ndvar,
        'mel': mel_ndvar,
    }

Song 1/10
Song 2/10
Song 3/10
Song 4/10
Song 5/10
Song 6/10
Song 7/10
Song 8/10
Song 9/10
Song 10/10


In [3]:
# Save all features

out_path = DATA_ROOT / 'song_features.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(song_features, f)
print('Features saved')

Features saved
